In [11]:
import os
os.listdir("/content")

['.config', 'flyrank-ml-internship', 'sample_data']

In [12]:
!git clone https://github.com/gitWithPrince272/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [13]:
import os
os.chdir("/content/flyrank-ml-internship")
print(os.getcwd())

/content/flyrank-ml-internship


In [14]:
import pandas as pd
import glob

csv_files = glob.glob("data/raw/*.csv")
print(csv_files)

df = pd.read_csv(csv_files[0])
df.head()

['data/raw/content_refresh_anonymized.csv']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gitWithPrince272/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

# Research Question

How can machine learning models identify and prioritize website pages that have opportunities for SEO improvement?

This supports the decision of which pages should be reviewed first by content and SEO teams.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

# Data

The analysis uses the FlyRank anonymized content dataset.

The dataset contains page-level SEO information including:
- impressions
- clicks
- CTR
- average position
- engagement metrics
- content-related features

Excluded information:
- Private client information
- Future data that could cause leakage
- Any information not available during the decision period

The data is used only for analysis and decision-support purposes.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

# Methodology

The workflow includes:

1. Data preparation and feature selection.
2. Building a baseline ranking approach.
3. Training a Decision Tree Regression model.
4. Evaluating performance using MAE and R² score.
5. Performing validation and leakage checks.

The model uses historical SEO signals to identify pages that may benefit from optimization.

Human review is required before taking any action.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

# Results

The Decision Tree model was evaluated against the validation split using the same dataset.

Model performance:
- Mean Absolute Error (MAE): 13.9047
- R² Score: 0.4541

The model captures some relationship between SEO features and the target variable. However, results should be considered a baseline and not a final production solution.

In [15]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import pandas as pd

# Select numerical columns
numeric_df = df.select_dtypes(include="number").copy()

# Fill missing values
numeric_df = numeric_df.fillna(numeric_df.median())

# Target variable
target = "trend_pct"

# Features and target
X = numeric_df.drop(columns=[target])
y = numeric_df[target]

# Same split as previous model
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train model
model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Results table
results = pd.DataFrame({
    "Metric": ["Mean Absolute Error", "R2 Score"],
    "Value": [round(mae,4), round(r2,4)]
})

display(results)

,Metric,Value
0,Mean Absolute Error,13.9047
1,R2 Score,0.4541


## 5. Limitations

*What this work cannot claim.*

# Limitations

This work has several limitations:

- The model uses historical SEO data and cannot predict future search trends.
- External factors such as Google algorithm updates and competitor actions are not included.
- The model output requires human review before applying content changes.
- Performance may change when evaluated on different groups or future data.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

# Ranked Recommendations

The action playbook prioritizes pages based on SEO opportunity signals.

Recommended actions:
1. Review pages with high impressions and low CTR.
2. Improve content quality and alignment with search intent.
3. Monitor ranking changes after updates.
4. Prioritize pages using model scores along with human judgment.

The recommendations are decision-support suggestions and should not be automated without review.

In [16]:
import pandas as pd

# Create recommendation score
recommendation_df = df.copy()

# Priority score based on SEO opportunity
recommendation_df["action_score"] = (
    (recommendation_df["impressions_90d"] / recommendation_df["impressions_90d"].max()) * 0.5
    +
    ((100 - recommendation_df["ctr"]) / 100) * 0.3
    +
    ((20 - recommendation_df["avg_position"].clip(lower=0)) / 20) * 0.2
)

# Rank pages
ranked_actions = recommendation_df.sort_values(
    by="action_score",
    ascending=False
)

# Show top recommendations
display(
    ranked_actions[
        [
            "content_id",
            "impressions_90d",
            "ctr",
            "avg_position",
            "action_score"
        ]
    ].head(10)
)

,content_id,impressions_90d,ctr,avg_position,action_score
26844,content_8c19996aa890,509252,0.15,2.5,0.966377
6653,content_5fe46e04994d,517715,0.14,4.2,0.957580
17812,content_aaef01a50def,517109,0.25,5.4,0.944665
21819,content_4c36c775b818,463103,0.41,2.3,0.923027
29879,content_1a9e894be2e2,416180,0.23,4.0,0.861249
13537,content_2c2606c5d176,347399,0.53,4.2,0.791922
14090,content_44e481c8f55b,312694,0.65,1.4,0.786044
18870,content_db5989a78dd3,345111,0.21,5.4,0.778672
21565,content_9532f197bbc8,309192,0.87,2.0,0.776002
19636,content_2cb567c3c89b,497727,0.10,22.2,0.758396


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

# Artifacts

The paper includes the following supporting artifacts:

- Model evaluation metrics (MAE and R² score).
- Ranked action queue generated from the playbook.
- Validation and leakage audit results.
- Exported CSV files used for recommendations.

These artifacts provide transparency and support reproducibility.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
